# Getting Started with DTL Python Bindings

This notebook introduces the basic concepts and usage of DTL (Distributed Template Library) Python bindings.

## What is DTL?

DTL is a C++ library for distributed computing that provides:
- Distributed containers (vectors, tensors)
- Automatic data partitioning across processes
- MPI-based communication
- Optional GPU acceleration

The Python bindings provide a NumPy-compatible interface to DTL.

In [ ]:
# Import DTL and check version
import dtl
import numpy as np

print(f"DTL version: {dtl.__version__}")
print(f"Version info: {dtl.version_info}")

## Feature Detection

DTL can be built with different backends. Use the feature detection functions to check what's available.

In [ ]:
print("Available backends:")
print(f"  MPI:  {dtl.has_mpi()}")
print(f"  CUDA: {dtl.has_cuda()}")
print(f"  HIP:  {dtl.has_hip()}")
print(f"  NCCL: {dtl.has_nccl()}")

## Creating a Context

All DTL operations require a `Context` object. The context encapsulates:
- MPI communicator (for distributed execution)
- Device selection (for GPU execution)

Use the context manager protocol for automatic cleanup:

In [ ]:
# Create a default context
with dtl.Context() as ctx:
    print(f"Rank: {ctx.rank}")
    print(f"Size: {ctx.size}")
    print(f"Is root: {ctx.is_root}")
    print(f"Device ID: {ctx.device_id}")
    print(f"Has device: {ctx.has_device}")

## Context with mpi4py

If mpi4py is available, you can pass an MPI communicator:

In [ ]:
try:
    from mpi4py import MPI
    
    # Use MPI_COMM_WORLD
    with dtl.Context(comm=MPI.COMM_WORLD) as ctx:
        print(f"MPI rank: {ctx.rank}")
        print(f"MPI size: {ctx.size}")
except ImportError:
    print("mpi4py not available")

## Creating a Distributed Vector

The simplest distributed container is `DistributedVector`. It automatically partitions data across ranks.

In [ ]:
with dtl.Context() as ctx:
    # Create a vector with 1000 elements
    vec = dtl.DistributedVector(ctx, size=1000, dtype=np.float64)
    
    print(f"Global size: {vec.global_size}")
    print(f"Local size: {vec.local_size}")
    print(f"Local offset: {vec.local_offset}")

## Zero-Copy NumPy Integration

The `local_view()` method returns a NumPy array that shares memory with the vector. This means:
- No data copying
- Modifications to the array modify the vector
- Full NumPy compatibility

In [ ]:
with dtl.Context() as ctx:
    vec = dtl.DistributedVector(ctx, size=100, dtype=np.float64)
    
    # Get local view
    local = vec.local_view()
    print(f"Type: {type(local)}")
    print(f"Shape: {local.shape}")
    print(f"Dtype: {local.dtype}")
    
    # Fill with NumPy operations
    local[:] = np.arange(len(local))
    print(f"\nFirst 10 elements: {local[:10]}")

## Synchronization

Use `barrier()` to synchronize all ranks:

In [ ]:
with dtl.Context() as ctx:
    vec = dtl.DistributedVector(ctx, size=100)
    local = vec.local_view()
    local[:] = ctx.rank
    
    # Synchronize all ranks
    ctx.barrier()
    
    print(f"Rank {ctx.rank} done")

## Next Steps

- [02_distributed_vectors.ipynb](02_distributed_vectors.ipynb) - More vector operations
- [03_distributed_tensors.ipynb](03_distributed_tensors.ipynb) - N-dimensional arrays
- [04_algorithms.ipynb](04_algorithms.ipynb) - Distributed algorithms